Limpieza y preparacion de los datos
este notebook documenta la carga del dataset bruto, el filtrado de valores atipicos o errores y la exportacion del dataset limpio

1. Filtrado de Inconsistencias y Valores Fuera de Rango
Se eliminan límites de velocidad inválidos (manteniendo el rango legal de 20 a 70 mph) y registros no asignados en el entorno geográfico.

In [10]:
import pandas as pd

INPUT_PATH = "../data/raw/datos_colisiones_granbretaña_2021-2025.csv"
OUTPUT_PATH = "../data/processed/datos_colisiones_procesados.csv"

print("Cargando dataset bruto")
cols = [
    'collision_index', 'collision_severity', 'collision_year', 'date', 'time',
    'day_of_week', 'weather_conditions', 'light_conditions', 'speed_limit',
    'road_type', 'road_surface_conditions', 'urban_or_rural_area'
]

df = pd.read_csv(INPUT_PATH, usecols=cols)
print(f"Total registros leidos: {len(df):,}")

print("Limpiando valores inconsistentes...")
df = df[df['speed_limit'].between(20, 70)]
df = df[df['urban_or_rural_area'].isin([1, 2])]
df = df[df['collision_year'].between(2021, 2025)]
df = df.dropna(subset=['date', 'time'])

Cargando dataset bruto
Total registros leidos: 513,801
Limpiando valores inconsistentes...


C:\Users\marcelo\AppData\Local\Temp\ipykernel_13128\1381839313.py:13: DtypeWarning: Columns (0: collision_index) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_PATH, usecols=cols)


## 2. Decodificación de Categorías según Guía Oficial STATS19
Mapeo de códigos numéricos a variables legibles para el análisis univariado y multivariado.

In [11]:
print("Decodificando variables...")
df['severidad'] = df['collision_severity'].map({1: 'Fatal', 2: 'Grave', 3: 'Leve'})
df['es_grave_fatal'] = df['collision_severity'].isin([1, 2]).astype(int)
df['es_fatal'] = (df['collision_severity'] == 1).astype(int)

df['entorno'] = df['urban_or_rural_area'].map({1: 'Urbano', 2: 'Rural'})

df['tipo_via'] = df['road_type'].map({
    1: 'Rotonda',
    2: 'Sentido único',
    3: 'Doble calzada',
    6: 'Calzada simple',
    7: 'Rampa de acceso'
}).fillna('Otro / Desconocido')

df['estado_calzada'] = df['road_surface_conditions'].map({
    1: 'Seco',
    2: 'Mojado / Húmedo',
    3: 'Nieve',
    4: 'Hielo / Escarcha',
    5: 'Inundado'
}).fillna('Otro / Desconocido')

df['clima'] = df['weather_conditions'].map({
    1: 'Despejado',
    2: 'Lluvia',
    3: 'Nieve',
    4: 'Despejado c/ viento',
    5: 'Lluvia c/ viento',
    6: 'Nieve c/ viento',
    7: 'Niebla'
}).fillna('Otro / Desconocido')

df['iluminacion'] = df['light_conditions'].map({
    1: 'Luz diurna',
    4: 'Noche (con alumbrado)',
    5: 'Noche (alumbrado apagado)',
    6: 'Noche (sin alumbrado)'
}).fillna('Otro / Desconocido')

df['dia_semana'] = df['day_of_week'].map({
    1: 'Domingo', 2: 'Lunes', 3: 'Martes', 4: 'Miércoles',
    5: 'Jueves', 6: 'Viernes', 7: 'Sábado'
})

df['fecha'] = pd.to_datetime(df['date'], format='%d/%m/%Y', errors='coerce')
df['hora'] = pd.to_datetime(df['time'], format='%H:%M', errors='coerce').dt.hour

columnas_procesadas = [
    'collision_index', 'collision_year', 'fecha', 'hora', 'dia_semana',
    'severidad', 'es_grave_fatal', 'es_fatal',
    'speed_limit', 'entorno', 'tipo_via', 'estado_calzada', 'clima', 'iluminacion'
]

Decodificando variables...


## 3. Exportación de Datos Procesados
Se guarda el conjunto de datos limpio en `data/processed/datos_colisiones_procesados.csv` para consumo directo de la aplicación Streamlit.

In [12]:
df_limpio = df[columnas_procesadas]

print(f"Guardando {len(df_limpio):,} registros en: {OUTPUT_PATH}")
df_limpio.to_csv(OUTPUT_PATH, index=False)
print("Limpieza completada.")

Guardando 513,740 registros en: ../data/processed/datos_colisiones_procesados.csv
Limpieza completada.
